In [1]:
# ==== Compare WITH anomalies vs AFTER DROPPING anomalies (same output structure) ====
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score, mean_absolute_error,
    precision_score, recall_score, f1_score, average_precision_score
)
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    IsolationForest, RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    VotingRegressor, StackingRegressor
)
from sklearn.linear_model import LinearRegression, Ridge, BayesianRidge, HuberRegressor
from sklearn.tree import DecisionTreeRegressor

# ---- Optional XGBoost (fallback to GBR if not available) ----
USE_XGB = True
try:
    from xgboost import XGBRegressor
except Exception as e:
    print("[WARN] xgboost not available, falling back to GradientBoostingRegressor for 'XGB' slot:", e)
    USE_XGB = False

# ---------- 0) Load + minimal features ----------
PATH = r"C:\Users\israt\HD lab\My Project\Dataset\Airsage Dataset\NebraskaLakes_Cleaned_round.csv"
df = pd.read_csv(PATH, low_memory=False)

# core columns
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")
df["count"] = pd.to_numeric(df["count"], errors="coerce")      # target
# handle duration if present, else create a neutral column
if "duration" in df.columns:
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
else:
    df["duration"] = 0.0

# drop unusable rows
df = df.dropna(subset=["date","count","duration"]).copy()

# ---------- simple time features (date-only; no target leakage) ----------
df["month"] = df["date"].dt.month                     # 1..12
df["dow"] = df["date"].dt.dayofweek                   # 0=Mon..6=Sun
df["day"] = df["date"].dt.day                         # 1..31
df["week"] = df["date"].dt.isocalendar().week.astype(int)  # 1..53
df["is_weekend"] = (df["dow"] >= 5).astype(int)

def _month_to_season(m: int) -> int:
    # Winter(0)=Dec,Jan,Feb; Spring(1)=Mar,Apr,May; Summer(2)=Jun,Jul,Aug; Fall(3)=Sep,Oct,Nov
    if m in (12, 1, 2):  return 0
    if m in (3, 4, 5):   return 1
    if m in (6, 7, 8):   return 2
    return 3

df["season"] = df["month"].apply(lambda m: _month_to_season(int(m)))

# --- meta to print later (NO effect on modeling/splits) ---
meta_all = df[["poi_id", "poi", "date"]].copy() if {"poi_id","poi"}.issubset(df.columns) else df[["date"]].copy()
if "poi_id" not in meta_all.columns: meta_all["poi_id"] = -1
if "poi" not in meta_all.columns:    meta_all["poi"]    = "UNKNOWN"
meta_all["date_str"] = meta_all["date"].dt.strftime("%Y%m%d")

# --- feature matrix / target ---
feat_cols = [c for c in df.select_dtypes(include=[np.number]).columns
             if c not in {"count","anomaly_label","anomaly_score"}]
X_all = df[feat_cols].fillna(0).to_numpy(np.float32)
y_all = df["count"].to_numpy(np.float32)

# ---------- 1) Models to evaluate ----------
candidates = [
    ("rf",    RandomForestRegressor(n_estimators=600, max_depth=None,
                                    min_samples_split=2, min_samples_leaf=1,
                                    max_features="sqrt", n_jobs=-1, random_state=42)),
    ("xgb",   XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                           subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                           n_jobs=-1, random_state=42, tree_method="hist"))
             if USE_XGB else
             ("xgb",   GradientBoostingRegressor(random_state=42, n_estimators=350, learning_rate=0.05, max_depth=3)),
    ("etr",   ExtraTreesRegressor(n_estimators=600, max_depth=None,
                                  min_samples_leaf=1, max_features="sqrt",
                                  n_jobs=-1, random_state=42)),
    ("gbr",   GradientBoostingRegressor(random_state=42, n_estimators=300, learning_rate=0.05, max_depth=3)),
    ("dt",    DecisionTreeRegressor(max_depth=12, random_state=42)),
    ("lr",    LinearRegression()),
    ("ridge", Ridge(alpha=1.0, random_state=42)),
    ("bayes", BayesianRidge()),
    ("huber", HuberRegressor(epsilon=1.35)),
]

# ---------- 2) Helper: rank models on one split ----------
def rank_models_on_split(X_tr, X_te, y_tr, y_te, candidates, label):
    TH = float(np.median(y_tr))  # threshold for P/R/F1/PR-AUC
    rows, fitted = [], {}
    for name, mdl in candidates:
        m = clone(mdl)
        m.fit(X_tr, y_tr)
        fitted[name] = m
        pred = m.predict(X_te)

        r2  = r2_score(y_te, pred)
        mae = mean_absolute_error(y_te, pred)

        y_true = (y_te > TH).astype(int)
        y_hat  = (pred  > TH).astype(int)

        P  = precision_score(y_true, y_hat, zero_division=0)
        R  = recall_score(y_true, y_hat, zero_division=0)
        F1 = f1_score(y_true, y_hat, zero_division=0)
        PR = average_precision_score(y_true, pred)

        rows.append({"Scenario": label, "Model": name.upper(),
                     "P": P, "R": R, "F1": F1, "PR_AUC": PR, "R2": r2, "MAE": mae})

    rank_df = (pd.DataFrame(rows)
                 .sort_values(["F1","PR_AUC","R2","MAE","P","R"],
                              ascending=[False,False,False,True,False,False])
                 .reset_index(drop=True))
    return rank_df

# ---------- 3) Scenario A: WITH anomalies (no drop) ----------
X_tr_A, X_te_A, y_tr_A, y_te_A = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
rank_with = rank_models_on_split(X_tr_A, X_te_A, y_tr_A, y_te_A, candidates, label="WITH_ANOMALIES")

# ---------- 4) Scenario B: AFTER DROPPING anomalies (IsolationForest prefilter) ----------
PREFILTER_CONTAM = 0.10
Z = StandardScaler().fit_transform(X_all)  # unsupervised, no target leakage
gate = IsolationForest(contamination=PREFILTER_CONTAM, random_state=42).fit(Z)
keep_mask = (gate.predict(Z) == 1)  # 1=normal, -1=anomaly
X_clean, y_clean = X_all[keep_mask], y_all[keep_mask]

# align meta with cleaned rows (printing only; no modeling change)
meta_clean = meta_all[keep_mask].reset_index(drop=True)

print(f"Dropped {len(y_all) - len(y_clean)} anomalies "
      f"({100*(1 - len(y_clean)/len(y_all)):.1f}%) using IsolationForest prefilter.")

# Split Scenario B WITH meta (does not alter X/y permutation)
X_tr_B, X_te_B, y_tr_B, y_te_B, m_tr_B, m_te_B = train_test_split(
    X_clean, y_clean, meta_clean, test_size=0.2, random_state=42
)
rank_wo = rank_models_on_split(X_tr_B, X_te_B, y_tr_B, y_te_B, candidates, label="NO_ANOMALIES")

# ---------- 5) Print BOTH tables (identical structure) ----------
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

_fmt = lambda x: f"{x:.3f}"

print("\n=== RANKED (WITH anomalies) ===")
print(
    rank_with[["Model","P","R","F1","PR_AUC","R2","MAE"]]
    .to_string(index=False, float_format=_fmt)
)

print("\n=== RANKED (AFTER DROPPING anomalies) ===")
print(
    rank_wo[["Model","P","R","F1","PR_AUC","R2","MAE"]]
    .to_string(index=False, float_format=_fmt)
)

# ---------- 6) Optional: side-by-side comparison by Model ----------
cmp = (
    rank_with.merge(rank_wo, on="Model", suffixes=("_WITH","_NOANOM"))
    .loc[:, [
        "Model",
        "F1_WITH","F1_NOANOM",
        "PR_AUC_WITH","PR_AUC_NOANOM",
        "R2_WITH","R2_NOANOM",
        "MAE_WITH","MAE_NOANOM",
        "P_WITH","P_NOANOM","R_WITH","R_NOANOM"
    ]]
)

print("\n=== SIDE-BY-SIDE (same models) ===")
print(cmp.to_string(index=False, float_format=_fmt))


# ---------- 7) Ensembles (Top-3) + Top-3 hybrid avg ----------
# Take Top-3 models from NO_ANOMALIES ranking
top3_noanom = rank_wo["Model"].head(3).tolist()   # e.g., ["XGB","RF","ETR"]
top3 = list(dict.fromkeys(top3_noanom))
assert len(top3) == 3, f"Expected 3 unique models in top3, got {top3}"

# We'll ensemble on the NO_ANOMALIES split
X_tr, X_te, y_tr, y_te = X_tr_B, X_te_B, y_tr_B, y_te_B

# Map model name -> estimator from `candidates` (uppercase keys)
cand_map = {name.upper(): est for name, est in candidates}
for nm in top3:
    assert nm in cand_map, f"{nm} not in candidates: {list(cand_map)}"

# Base learners for ensembles (must be uppercase names)
bases = [(nm, clone(cand_map[nm])) for nm in top3]

# Small cache of fitted base candidates (keys are original names: 'rf','xgb',...)
fitted = {name: clone(est).fit(X_tr, y_tr) for name, est in candidates}

# Voting (equal weights across Top-3)
vote = VotingRegressor(bases).fit(X_tr, y_tr)
pred_vote = vote.predict(X_te)

# Stacking with several meta-learners
stack_lin = StackingRegressor(
    estimators=bases,
    final_estimator=LinearRegression(),
    cv=3, passthrough=True
).fit(X_tr, y_tr)
pred_stack_lin = stack_lin.predict(X_te)

stack_ridge = StackingRegressor(
    estimators=bases,
    final_estimator=Ridge(alpha=1.0, random_state=42),
    cv=3, passthrough=True
).fit(X_tr, y_tr)
pred_stack_ridge = stack_ridge.predict(X_te)

stack_gbr = StackingRegressor(
    estimators=bases,
    final_estimator=GradientBoostingRegressor(random_state=42, n_estimators=200, learning_rate=0.05, max_depth=3),
    cv=3, passthrough=True
).fit(X_tr, y_tr)
pred_stack_gbr = stack_gbr.predict(X_te)

# ---- Hybrid (Top-3 equal-weight average) ----
def _predict_from_name(nm_upper: str):
    key_l = nm_upper.lower()
    if key_l in fitted:
        return fitted[key_l].predict(X_te)
    return clone(cand_map[nm_upper]).fit(X_tr, y_tr).predict(X_te)

_top3_preds = [_predict_from_name(nm) for nm in top3]     # e.g., preds from ["XGB","RF","ETR"]
pred_avg = np.mean(np.column_stack(_top3_preds), axis=1)

# ---------- 8) Report ----------
def show(name, yt, yp):
    print(f"{name:<24} R2={r2_score(yt, yp):.4f}  MAE={mean_absolute_error(yt, yp):.4f}")

print("\n=== Ensembles with Top-3 bases (NO_ANOMALIES split) ===")
show("Voting (Top-3, equal)", y_te, pred_vote)
show("Stack (Linear meta)",   y_te, pred_stack_lin)
show("Stack (Ridge meta)",    y_te, pred_stack_ridge)
show("Stack (GBR meta)",      y_te, pred_stack_gbr)
show("Hybrid (Top-3 avg)",    y_te, pred_avg)

# ---------- 9) (Optional) Quick peek at some rows from NO_ANOMALIES test split ----------
N_SHOW = min(10, len(m_te_B))
print("\n-- Sample rows (NO_ANOMALIES test) --")
print("POI_ID | POI_NAME | DATE | Actual | Vote | StackLin | StackRidge | StackGBR | Hybrid")
for i in range(N_SHOW):
    print(f"{m_te_B.iloc[i]['poi_id']} | {m_te_B.iloc[i]['poi']} | {m_te_B.iloc[i]['date_str']} | "
          f"{int(round(y_te[i]))} | {int(round(pred_vote[i]))} | "
          f"{int(round(pred_stack_lin[i]))} | {int(round(pred_stack_ridge[i]))} | "
          f"{int(round(pred_stack_gbr[i]))} | {int(round(pred_avg[i]))}")


C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=2.68519e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Dropped 12363 anomalies (10.0%) using IsolationForest prefilter.


C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=2.23765e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)



=== RANKED (WITH anomalies) ===
Model     P     R    F1  PR_AUC    R2   MAE
   LR 0.987 0.999 0.993   1.000 1.000 0.379
   RF 0.987 0.998 0.992   1.000 0.981 5.411
  ETR 0.986 0.998 0.992   1.000 0.982 4.750
  GBR 0.993 0.988 0.991   1.000 0.991 6.958
HUBER 0.984 0.987 0.986   0.999 0.998 5.413
  XGB 0.982 0.985 0.984   0.999 0.985 6.280
BAYES 0.976 0.989 0.983   0.999 1.000 1.312
RIDGE 0.976 0.989 0.983   0.999 1.000 1.315
   DT 0.974 0.978 0.976   0.995 0.981 9.983

=== RANKED (AFTER DROPPING anomalies) ===
Model     P     R    F1  PR_AUC    R2   MAE
  ETR 0.979 0.999 0.989   1.000 0.998 1.292
   RF 0.978 0.998 0.988   1.000 0.998 1.484
   LR 0.974 0.999 0.986   1.000 1.000 0.382
  GBR 0.978 0.994 0.986   1.000 0.999 1.711
BAYES 0.973 0.998 0.985   0.999 1.000 0.803
RIDGE 0.973 0.998 0.985   0.999 1.000 0.806
  XGB 0.976 0.994 0.985   0.999 0.999 1.597
HUBER 0.972 0.992 0.982   0.999 0.999 1.834
   DT 0.964 0.986 0.975   0.997 0.997 2.669

=== SIDE-BY-SIDE (same models) ===
Model  F

KeyboardInterrupt: 

In [2]:
# ==== Compare WITH anomalies vs AFTER DROPPING anomalies (feature order frozen) ====
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score, mean_absolute_error,
    precision_score, recall_score, f1_score, average_precision_score
)
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    IsolationForest, RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    VotingRegressor, StackingRegressor
)
from sklearn.linear_model import LinearRegression, Ridge, BayesianRidge, HuberRegressor
from sklearn.tree import DecisionTreeRegressor

# ---- Optional XGBoost (fallback to GBR if not available) ----
USE_XGB = True
try:
    from xgboost import XGBRegressor
except Exception as e:
    print("[WARN] xgboost not available, falling back to GradientBoostingRegressor for 'XGB' slot:", e)
    USE_XGB = False

# ---------- 0) Load + minimal features ----------
PATH = r"C:\Users\israt\HD lab\My Project\Dataset\Airsage Dataset\NebraskaLakes_Cleaned_round.csv"
df = pd.read_csv(PATH, low_memory=False)

# core columns
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")
df["count"] = pd.to_numeric(df["count"], errors="coerce")      # target
if "duration" in df.columns:
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
else:
    df["duration"] = 0.0

# drop unusable rows
df = df.dropna(subset=["date","count","duration"]).copy()

# ---------- simple time features (date-only; no target leakage) ----------
df["month"] = df["date"].dt.month                     # 1..12
df["dow"] = df["date"].dt.dayofweek                   # 0=Mon..6=Sun
df["day"] = df["date"].dt.day                         # 1..31
df["week"] = df["date"].dt.isocalendar().week.astype(int)  # 1..53 (cast to int)
df["is_weekend"] = (df["dow"] >= 5).astype(int)

def _month_to_season(m: int) -> int:
    # Winter(0)=Dec,Jan,Feb; Spring(1)=Mar,Apr,May; Summer(2)=Jun,Jul,Aug; Fall(3)=Sep,Oct,Nov
    if m in (12, 1, 2):  return 0
    if m in (3, 4, 5):   return 1
    if m in (6, 7, 8):   return 2
    return 3

df["season"] = df["month"].apply(lambda m: _month_to_season(int(m)))

# --- meta to print later (NO effect on modeling/splits) ---
meta_all = df[["poi_id", "poi", "date"]].copy() if {"poi_id","poi"}.issubset(df.columns) else df[["date"]].copy()
if "poi_id" not in meta_all.columns: meta_all["poi_id"] = -1
if "poi" not in meta_all.columns:    meta_all["poi"]    = "UNKNOWN"
meta_all["date_str"] = meta_all["date"].dt.strftime("%Y%m%d")

# ---------- FREEZE FEATURE ORDER (one canonical list) ----------
TIME_FEATS = ["month", "dow", "day", "week", "is_weekend", "season"]

_num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for bad in ("count", "anomaly_label", "anomaly_score"):
    if bad in _num_cols:
        _num_cols.remove(bad)

# Put time features first (in the exact order above), then the rest sorted (stable & deterministic)
others = [c for c in _num_cols if c not in TIME_FEATS]
others.sort()
FEAT_COLS = tuple([c for c in TIME_FEATS if c in _num_cols] + others)

def build_X(frame: pd.DataFrame) -> np.ndarray:
    """Always build X with the frozen FEAT_COLS order; fill missing with 0. Cast week to int."""
    # create missing columns as zeros so shapes always match
    missing = [c for c in FEAT_COLS if c not in frame.columns]
    for c in missing:
        frame[c] = 0.0
    if "week" in frame.columns:
        frame["week"] = frame["week"].astype(int)
    X = frame.loc[:, FEAT_COLS].fillna(0).to_numpy(np.float32)
    return X

# Build training matrix with the frozen order
X_all = build_X(df)
y_all = df["count"].to_numpy(np.float32)

# ---------- 1 Models to evaluate ----------
candidates = [
    ("rf",    RandomForestRegressor(n_estimators=600, max_depth=None,
                                    min_samples_split=2, min_samples_leaf=1,
                                    max_features="sqrt", n_jobs=-1, random_state=42)),
    ("xgb",   XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                           subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                           n_jobs=-1, random_state=42, tree_method="hist"))
             if USE_XGB else
             ("xgb",   GradientBoostingRegressor(random_state=42, n_estimators=350, learning_rate=0.05, max_depth=3)),
    ("etr",   ExtraTreesRegressor(n_estimators=600, max_depth=None,
                                  min_samples_leaf=1, max_features="sqrt",
                                  n_jobs=-1, random_state=42)),
    ("gbr",   GradientBoostingRegressor(random_state=42, n_estimators=300, learning_rate=0.05, max_depth=3)),
    ("dt",    DecisionTreeRegressor(max_depth=12, random_state=42)),
    ("lr",    LinearRegression()),
    ("ridge", Ridge(alpha=1.0, random_state=42)),
    ("bayes", BayesianRidge()),
    ("huber", HuberRegressor(epsilon=1.35)),
]

# ---------- 2) Helper: rank models on one split ----------
def rank_models_on_split(X_tr, X_te, y_tr, y_te, candidates, label):
    TH = float(np.median(y_tr))  # threshold for P/R/F1/PR-AUC
    rows, fitted = [], {}
    for name, mdl in candidates:
        m = clone(mdl)
        m.fit(X_tr, y_tr)
        fitted[name] = m
        pred = m.predict(X_te)

        r2  = r2_score(y_te, pred)
        mae = mean_absolute_error(y_te, pred)

        y_true = (y_te > TH).astype(int)
        y_hat  = (pred  > TH).astype(int)

        P  = precision_score(y_true, y_hat, zero_division=0)
        R  = recall_score(y_true, y_hat, zero_division=0)
        F1 = f1_score(y_true, y_hat, zero_division=0)
        PR = average_precision_score(y_true, pred)

        rows.append({"Scenario": label, "Model": name.upper(),
                     "P": P, "R": R, "F1": F1, "PR_AUC": PR, "R2": r2, "MAE": mae})

    rank_df = (pd.DataFrame(rows)
                 .sort_values(["F1","PR_AUC","R2","MAE","P","R"],
                              ascending=[False,False,False,True,False,False])
                 .reset_index(drop=True))
    return rank_df

# ---------- 3) Scenario A: WITH anomalies (no drop) ----------
X_tr_A, X_te_A, y_tr_A, y_te_A = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
rank_with = rank_models_on_split(X_tr_A, X_te_A, y_tr_A, y_te_A, candidates, label="WITH_ANOMALIES")

# ---------- 4) Scenario B: AFTER DROPPING anomalies (IsolationForest prefilter) ----------
PREFILTER_CONTAM = 0.10
Z = StandardScaler().fit_transform(X_all)  # unsupervised, no target leakage
gate = IsolationForest(contamination=PREFILTER_CONTAM, random_state=42).fit(Z)
keep_mask = (gate.predict(Z) == 1)  # 1=normal, -1=anomaly

# Rebuild X/y from the same rows using the SAME FEAT_COLS order
df_clean = df.loc[keep_mask].copy()
X_clean = build_X(df_clean)
y_clean = y_all[keep_mask]

# align meta with cleaned rows (printing only; no modeling change)
meta_clean = meta_all[keep_mask].reset_index(drop=True)

print(f"Dropped {len(y_all) - len(y_clean)} anomalies "
      f"({100*(1 - len(y_clean)/len(y_all)):.1f}%) using IsolationForest prefilter.")

# Split Scenario B WITH meta (does not alter X/y permutation)
X_tr_B, X_te_B, y_tr_B, y_te_B, m_tr_B, m_te_B = train_test_split(
    X_clean, y_clean, meta_clean, test_size=0.2, random_state=42
)
rank_wo = rank_models_on_split(X_tr_B, X_te_B, y_tr_B, y_te_B, candidates, label="NO_ANOMALIES")

# ---------- 5) Print BOTH tables (identical structure) ----------
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

_fmt = lambda x: f"{x:.3f}"

print("\n=== RANKED (WITH anomalies) ===")
print(
    rank_with[["Model","P","R","F1","PR_AUC","R2","MAE"]]
    .to_string(index=False, float_format=_fmt)
)

print("\n=== RANKED (AFTER DROPPING anomalies) ===")
print(
    rank_wo[["Model","P","R","F1","PR_AUC","R2","MAE"]]
    .to_string(index=False, float_format=_fmt)
)

# ---------- 6) Optional: side-by-side comparison by Model ----------
cmp = (
    rank_with.merge(rank_wo, on="Model", suffixes=("_WITH","_NOANOM"))
    .loc[:, [
        "Model",
        "F1_WITH","F1_NOANOM",
        "PR_AUC_WITH","PR_AUC_NOANOM",
        "R2_WITH","R2_NOANOM",
        "MAE_WITH","MAE_NOANOM",
        "P_WITH","P_NOANOM","R_WITH","R_NOANOM"
    ]]
)

print("\n=== SIDE-BY-SIDE (same models) ===")
print(cmp.to_string(index=False, float_format=_fmt))

# ---------- 7) Ensembles (Top-3) + Top-3 hybrid avg ----------
# Take Top-3 models from NO_ANOMALIES ranking
top3_noanom = rank_wo["Model"].head(3).tolist()   # e.g., ["XGB","RF","ETR"]
top3 = list(dict.fromkeys(top3_noanom))
assert len(top3) == 3, f"Expected 3 unique models in top3, got {top3}"

# We'll ensemble on the NO_ANOMALIES split
X_tr, X_te, y_tr, y_te = X_tr_B, X_te_B, y_tr_B, y_te_B

# Map model name -> estimator from `candidates` (uppercase keys)
cand_map = {name.upper(): est for name, est in candidates}
for nm in top3:
    assert nm in cand_map, f"{nm} not in candidates: {list(cand_map)}"

# Base learners for ensembles (must be uppercase names)
bases = [(nm, clone(cand_map[nm])) for nm in top3]

# Small cache of fitted base candidates (keys are original names: 'rf','xgb',...)
fitted = {name: clone(est).fit(X_tr, y_tr) for name, est in candidates}

# Voting (equal weights across Top-3)
vote = VotingRegressor(bases).fit(X_tr, y_tr)
pred_vote = vote.predict(X_te)

# Stacking with several meta-learners
stack_lin = StackingRegressor(
    estimators=bases,
    final_estimator=LinearRegression(),
    cv=3, passthrough=True
).fit(X_tr, y_tr)
pred_stack_lin = stack_lin.predict(X_te)

stack_ridge = StackingRegressor(
    estimators=bases,
    final_estimator=Ridge(alpha=1.0, random_state=42),
    cv=3, passthrough=True
).fit(X_tr, y_tr)
pred_stack_ridge = stack_ridge.predict(X_te)

stack_gbr = StackingRegressor(
    estimators=bases,
    final_estimator=GradientBoostingRegressor(random_state=42, n_estimators=200, learning_rate=0.05, max_depth=3),
    cv=3, passthrough=True
).fit(X_tr, y_tr)
pred_stack_gbr = stack_gbr.predict(X_te)

# ---- Hybrid (Top-3 equal-weight average) ----
def _predict_from_name(nm_upper: str):
    key_l = nm_upper.lower()
    if key_l in fitted:
        return fitted[key_l].predict(X_te)
    return clone(cand_map[nm_upper]).fit(X_tr, y_tr).predict(X_te)

_top3_preds = [_predict_from_name(nm) for nm in top3]     # e.g., preds from ["XGB","RF","ETR"]
pred_avg = np.mean(np.column_stack(_top3_preds), axis=1)

# ---------- 8) Report ----------
def show(name, yt, yp):
    print(f"{name:<24} R2={r2_score(yt, yp):.4f}  MAE={mean_absolute_error(yt, yp):.4f}")

print("\n=== Ensembles with Top-3 bases (NO_ANOMALIES split) ===")
show("Voting (Top-3, equal)", y_te, pred_vote)
show("Stack (Linear meta)",   y_te, pred_stack_lin)
show("Stack (Ridge meta)",    y_te, pred_stack_ridge)
show("Stack (GBR meta)",      y_te, pred_stack_gbr)
show("Hybrid (Top-3 avg)",    y_te, pred_avg)




C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=2.67562e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Dropped 12363 anomalies (10.0%) using IsolationForest prefilter.


C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=2.06587e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)



=== RANKED (WITH anomalies) ===
Model     P     R    F1  PR_AUC    R2   MAE
   LR 0.987 0.999 0.993   1.000 1.000 0.379
  ETR 0.987 0.998 0.993   1.000 0.982 4.765
RIDGE 0.988 0.997 0.992   1.000 1.000 0.644
BAYES 0.987 0.997 0.992   1.000 1.000 0.662
   RF 0.986 0.998 0.992   1.000 0.981 5.405
  GBR 0.993 0.988 0.991   1.000 0.991 6.953
  XGB 0.986 0.988 0.987   0.999 0.986 5.854
HUBER 0.984 0.987 0.986   0.999 0.998 5.412
   DT 0.974 0.978 0.976   0.995 0.981 9.879

=== RANKED (AFTER DROPPING anomalies) ===
Model     P     R    F1  PR_AUC    R2   MAE
  ETR 0.978 0.998 0.988   1.000 0.998 1.231
   RF 0.979 0.997 0.988   1.000 0.998 1.384
  XGB 0.980 0.994 0.987   0.999 0.999 1.379
   LR 0.974 0.999 0.986   1.000 1.000 0.388
  GBR 0.983 0.988 0.985   0.999 0.999 1.734
BAYES 0.970 0.999 0.984   1.000 1.000 0.421
RIDGE 0.970 0.999 0.984   1.000 1.000 0.426
   DT 0.965 0.985 0.975   0.997 0.997 2.702
HUBER 0.969 0.965 0.967   0.995 0.994 3.846

=== SIDE-BY-SIDE (same models) ===
Model  F

C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=2.06587e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)



=== Ensembles with Top-3 bases (NO_ANOMALIES split) ===
Voting (Top-3, equal)    R2=0.9989  MAE=1.1522
Stack (Linear meta)      R2=0.9999  MAE=0.4083
Stack (Ridge meta)       R2=0.9999  MAE=0.4083
Stack (GBR meta)         R2=0.9995  MAE=1.1074
Hybrid (Top-3 avg)       R2=0.9989  MAE=1.1522


In [3]:
# ---------- 9) (Optional) Quick peek at some rows from NO_ANOMALIES test split ----------
N_SHOW = min(10, len(m_te_B))
print("\n-- Sample rows (NO_ANOMALIES test) --")
print("POI_ID | POI_NAME | DATE | Actual | Vote | StackLin | StackRidge | StackGBR | Hybrid")
for i in range(N_SHOW):
    print(f"{m_te_B.iloc[i]['poi_id']} | {m_te_B.iloc[i]['poi']} | {m_te_B.iloc[i]['date_str']} | "
          f"{int(round(y_te[i]))} | {int(round(pred_vote[i]))} | "
          f"{int(round(pred_stack_lin[i]))} | {int(round(pred_stack_ridge[i]))} | "
          f"{int(round(pred_stack_gbr[i]))} | {int(round(pred_avg[i]))}")


-- Sample rows (NO_ANOMALIES test) --
POI_ID | POI_NAME | DATE | Actual | Vote | StackLin | StackRidge | StackGBR | Hybrid
6525.0 | Gallagher Canyon | 20220108 | 24 | 24 | 25 | 25 | 24 | 24
5710.0 | Carter Lake | 20220923 | 43 | 41 | 44 | 44 | 41 | 41
5745.0 | Holmes | 20220902 | 20 | 20 | 19 | 19 | 20 | 20
6915.0 | Harlan County Reservoir (COE) | 20221201 | 54 | 54 | 54 | 54 | 56 | 54
6707.0 | Heartwell Park | 20220929 | 78 | 75 | 77 | 77 | 76 | 75
6080.0 | Mormon Island West (MM 312N) | 20221117 | 15 | 15 | 15 | 15 | 15 | 15
4540.0 | Lake McConaughy | 20220428 | 22 | 22 | 22 | 22 | 22 | 22
5745.0 | Holmes | 20220701 | 25 | 25 | 25 | 25 | 25 | 25
3710.0 | Lewis & Clark Lake | 20240618 | 440 | 439 | 441 | 441 | 437 | 439
6716.0 | GI Pier Park Lake | 20220119 | 21 | 21 | 22 | 22 | 21 | 21


In [4]:
# ================== SIMPLE (tunable): stronger weekend/summer separation ==================
import numpy as np, pandas as pd
from sklearn.base import clone

def forecast_compare_simple(
    df_full, feat_cols, poi_id=None, poi_name=None,
    start_date=None, days=7,
    which="hybrid",                 # 'ridge'|'lin'|'gbr'|'voting'|'hybrid'|'all'
    round_int=False,
    season_boost=True,              # turn on/off seasonality multiplier for WITH_EXTRAS
    season_mode="product",          # 'product' (strong) or 'blend' (soft)
    w_month=0.75,                   # weight for month in 'blend' mode
    weekend_power=1.2,              # raise DOW factor to this power (>=1 increases weekend lift)
    month_power=1.1,                # raise MONTH factor to this power (>=1 increases summer lift)
    cap=(0.6, 1.6)                  # clamp season factor to [min,max] to avoid extremes
):
    # ---- tiny helpers ----
    def _poi(df, pid, pname):
        if pid is not None:
            s = df[df["poi_id"]==int(pid)]
            return int(pid), (s["poi"].iloc[0] if not s.empty and "poi" in s else str(pid))
        key = str(pname).strip().upper()
        s = df[df["poi"].astype(str).str.upper()==key]
        if s.empty: s = df[df["poi"].astype(str).str.upper().str.startswith(key)]
        if s.empty: raise ValueError(f"POI '{pname}' not found.")
        return int(s["poi_id"].iloc[0]), str(s["poi"].iloc[0])

    def _last(df, pid, asof=None):
        d = df[df["poi_id"]==pid].sort_values("date")
        if d.empty: raise ValueError(f"No history for poi_id={pid}.")
        if asof is not None:
            d2 = d[d["date"]<=asof]
            if not d2.empty: d = d2
        return d.iloc[-1], d

    # trained column layout
    probe = None
    for nm in ("stack_ridge","stack_lin","stack_gbr","vote"):
        if nm in globals():
            probe = globals()[nm]; break
    trained_cols = list(getattr(probe, "feature_names_in_", feat_cols))

    # predictors
    def _predict(Xf):
        preds = {}
        if which in ("ridge","all") and "stack_ridge" in globals(): preds["Stack_Ridge"]=float(stack_ridge.predict(Xf)[0])
        if which in ("lin","all")   and "stack_lin"   in globals(): preds["Stack_Lin"]  =float(stack_lin.predict(Xf)[0])
        if which in ("gbr","all")   and "stack_gbr"   in globals(): preds["Stack_GBR"]  =float(stack_gbr.predict(Xf)[0])
        if which in ("voting","all")and "vote"        in globals(): preds["Voting"]     =float(vote.predict(Xf)[0])
        if which in ("hybrid","all"):
            if "top3" not in globals(): raise RuntimeError("Need global 'top3' from section 7.")
            hy=[]
            for nm in top3:
                mdl = globals()["fitted"].get(nm.lower())
                if mdl is None: mdl = clone(globals()["cand_map"][nm]).fit(globals()["X_tr"], globals()["y_tr"])
                hy.append(float(mdl.predict(Xf)[0]))
            preds["Hybrid"]=float(np.mean(hy))
        if not preds: raise RuntimeError("No fitted model available.")
        return (float(np.mean(list(preds.values()))) if which=="all"
                else (preds.get("Hybrid") or preds.get("Voting") or preds.get("Stack_GBR") or preds.get("Stack_Ridge") or preds.get("Stack_Lin")))

    # feature row builders
    def _row_with(last_row, cur, last_count, recent):
        x = {c:0.0 for c in trained_cols}
        for c in trained_cols:
            if c in last_row.index:
                try: x[c]=float(last_row[c])
                except: pass
        if "poi_id" in x: x["poi_id"]=float(last_row["poi_id"])
        # calendar + Fourier
        if "month" in x: x["month"]=float(cur.month)
        if "dow" in x: x["dow"]=float(cur.dayofweek)
        if "is_weekend" in x: x["is_weekend"]=float(int(cur.dayofweek in (5,6)))
        if ("sin_y" in x) or ("cos_y" in x):
            doy=float(cur.timetuple().tm_yday)
            if "sin_y" in x: x["sin_y"]=float(np.sin(2*np.pi*doy/365.0))
            if "cos_y" in x: x["cos_y"]=float(np.cos(2*np.pi*doy/365.0))
        # lags
        if "lag_1" in x: x["lag_1"]=float(last_count)
        if "lag_7" in x: x["lag_7"]=float(recent[0] if len(recent)>=7 else last_count)
        if "roll7" in x and len(recent)>0: x["roll7"]=float(np.mean(recent))
        return pd.DataFrame([[x[c] for c in trained_cols]], columns=trained_cols)

    def _row_noextras(last_row):
        x = {c:0.0 for c in trained_cols}
        for c in trained_cols:
            if c in last_row.index:
                try: x[c]=float(last_row[c])
                except: pass
        if "poi_id" in x: x["poi_id"]=float(last_row["poi_id"])
        # intentionally DO NOT update calendar/Fourier/lags
        return pd.DataFrame([[x[c] for c in trained_cols]], columns=trained_cols)

    # season profiles (data-driven)
    def _season_profiles(hist):
        gmed = float(hist["count"].median()) or 1.0
        dow_rel = (hist.groupby(hist["date"].dt.dayofweek)["count"].median()/gmed).to_dict()
        mon_rel = (hist.groupby(hist["date"].dt.month)["count"].median()/gmed).to_dict()
        return dow_rel, mon_rel

    def _season_factor(dow_rel, mon_rel, dt):
        d = int(dt.dayofweek); m = int(dt.month)
        d_fac = float(dow_rel.get(d,1.0))
        m_fac = float(mon_rel.get(m,1.0))
        if season_mode == "blend":
            fac = (1.0 - w_month)*d_fac + w_month*m_fac
        else:  # 'product' (stronger)
            fac = (d_fac**weekend_power) * (m_fac**month_power)
        # clamp
        lo, hi = cap
        return max(lo, min(hi, fac))

    # -------- run both modes --------
    pid, pname = _poi(df_full, poi_id, poi_name)
    if start_date is None:
        last_d = df_full[df_full["poi_id"]==pid]["date"].max()
        if pd.isna(last_d): raise ValueError(f"No history for poi_id={pid}.")
        start = (last_d + pd.Timedelta(days=1)).normalize()
    else:
        start = pd.to_datetime(str(start_date), errors="coerce")
        if pd.isna(start): raise ValueError("Bad start_date; use YYYY-MM-DD or YYYYMMDD")

    last_row, hist = _last(df_full, pid, asof=start - pd.Timedelta(days=1))
    last_count0 = float(last_row["count"])
    before = hist[hist["date"] <= (start - pd.Timedelta(days=1))]
    recent0 = before["count"].tail(7).astype(float).tolist() if not before.empty else [last_count0]
    if not recent0: recent0=[last_count0]

    # season dicts
    dow_rel, mon_rel = _season_profiles(hist)

    outputs=[]
    for mode in ("WITH_EXTRAS","NO_EXTRAS"):
        rows=[]; cur=start; last_row_m=last_row.copy(); last_count=last_count0; recent=list(recent0)
        for _ in range(int(days)):
            Xf = _row_with(last_row_m, cur, last_count, recent) if mode=="WITH_EXTRAS" else _row_noextras(last_row_m)
            yhat = _predict(Xf)
            if mode=="WITH_EXTRAS" and season_boost:
                yhat *= _season_factor(dow_rel, mon_rel, cur)
            rows.append({
                "mode":mode,"poi_id":pid,"poi":pname,"date":cur,"date_str":cur.strftime("%Y-%m-%d"),
                "predicted": int(round(yhat)) if round_int else float(yhat)
            })
            if mode=="WITH_EXTRAS":
                recent.append(yhat); 
                if len(recent)>7: recent.pop(0)
            last_count=yhat
            last_row_m = last_row_m.copy()
            if "count" in last_row_m.index: last_row_m["count"]=last_count
            cur += pd.Timedelta(days=1)
        outputs.append(pd.DataFrame(rows))
    out = pd.concat(outputs, ignore_index=True)
    return out[["mode","poi_id","poi","date","date_str","predicted"]]


In [5]:
print(forecast_compare_simple(df, feat_cols, poi_name="Carter Lake", start_date="2025-01-01", days=7,
                               which="hybrid", season_boost=True, season_mode="product",
                               weekend_power=1.4, month_power=1.2, cap=(0.7, 1.7))
      .pivot(index="date_str", columns="mode", values="predicted").to_string())


C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but ExtraTreesRegressor was fitted without feature names
  warnings.warn(
C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but ExtraTreesRegressor was fitted without feature names
  warnings.warn(
C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:486: UserWarning: X has feature names, 

mode         NO_EXTRAS  WITH_EXTRAS
date_str                           
2025-01-01  210.060012   183.389732
2025-01-02  210.060012   159.977095
2025-01-03  210.060012   167.355475
2025-01-04  210.060012   167.028111
2025-01-05  210.060012   167.106384
2025-01-06  210.060012   184.962938
2025-01-07  210.060012   168.670419


C:\Users\israt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [14]:
import pandas as pd
import numpy as np

def simple_forecast(
    df_full: pd.DataFrame,
    poi_id: int | None = None,
    poi_name: str | None = None,
    start_date: str | None = None,
    days: int = 7,
    recent_window: int = 14,         # scale to recent level
    weekend_floor: float = 1.15,     # ensure weekends >= +15%
    summer_floor: float = 1.20,      # ensure Jun–Aug >= +20%
    weekday_floor: float = 0.95,     # weekdays at least 95% of baseline
    shoulder_floor: float = 1.00,    # non-summer months at least 100%
    dow_power: float = 1.0,          # amplify DOW profile
    month_power: float = 1.0,        # amplify month profile
    level_cap: tuple[float,float] = (0.3, 3.0),  # clamp recent level vs global median
    season_cap: tuple[float,float] = (0.5, 2.5), # clamp combined seasonal multiplier
    round_int: bool = False
) -> pd.DataFrame:
    # ---- checks ----
    need = {"date","count","poi_id"}
    if not need.issubset(df_full.columns):
        raise ValueError(f"DataFrame must contain columns: {need}")

    # ---- POI resolution ----
    if poi_id is None:
        key = str(poi_name).strip().upper()
        sub = df_full[df_full.get("poi","").astype(str).str.upper() == key]
        if sub.empty and "poi" in df_full:
            sub = df_full[df_full["poi"].astype(str).str.upper().str.startswith(key)]
        if sub.empty:
            raise ValueError(f"POI '{poi_name}' not found.")
        poi_id = int(sub["poi_id"].iloc[0])
        poi_name = str(sub["poi"].iloc[0]) if "poi" in df_full else str(poi_id)
    else:
        s = df_full[df_full["poi_id"] == int(poi_id)]
        poi_name = s["poi"].iloc[0] if "poi" in df_full and not s.empty else str(poi_id)

    # ---- history for POI ----
    d = (df_full[df_full["poi_id"] == int(poi_id)]
           .dropna(subset=["date","count"])
           .copy())
    if d.empty:
        raise ValueError("No history for that POI.")
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values("date")
    d["month"] = d["date"].dt.month
    d["dow"]   = d["date"].dt.dayofweek

    # global median and relative profiles
    gmed = float(d["count"].median())
    if gmed == 0: gmed = 1.0
    dow_rel = (d.groupby("dow")["count"].median() / gmed).to_dict()
    mon_rel = (d.groupby("month")["count"].median() / gmed).to_dict()

    # start date
    if start_date is None:
        start = d["date"].max() + pd.Timedelta(days=1)
    else:
        start = pd.to_datetime(str(start_date), errors="coerce")
        if pd.isna(start):
            raise ValueError("Invalid start_date; use 'YYYY-MM-DD' or 'YYYYMMDD'.")

    # recent level factor
    before = d[d["date"] <= (start - pd.Timedelta(days=1))]
    recent_vals = (before if not before.empty else d)["count"].tail(int(recent_window))
    recent_mean = float(recent_vals.mean()) if not recent_vals.empty else gmed
    level = recent_mean / gmed
    level = float(np.clip(level, level_cap[0], level_cap[1]))

    weekend_set = {5, 6}
    summer_set  = {6, 7, 8}

    # ---- forecast (no recursion) ----
    rows = []
    cur = start
    for _ in range(int(days)):
        m = int(cur.month); dow = int(cur.dayofweek)

        # base relative factors (fallback to 1.0 if missing), with power
        dfac = float(dow_rel.get(dow, 1.0)) ** float(dow_power)
        mfac = float(mon_rel.get(m, 1.0))   ** float(month_power)

        # enforce floors
        dfac = max(dfac, weekend_floor if dow in weekend_set else weekday_floor)
        mfac = max(mfac, summer_floor  if m   in summer_set  else shoulder_floor)

        season = dfac * mfac
        season = float(np.clip(season, season_cap[0], season_cap[1]))

        yhat = level * gmed * season
        if round_int:
            yhat = int(round(yhat))

        rows.append({
            "poi_id": int(poi_id),
            "poi": poi_name,
            "date": cur,
            "day": cur.strftime("%A"),
            "predicted": yhat
        })
        cur += pd.Timedelta(days=1)

    return pd.DataFrame(rows, columns=["poi_id","poi","date","day","predicted"])


In [17]:
print(simple_forecast(df, poi_name="Carter Lake", start_date="2025-07-01", days=7).to_string(index=False))

 poi_id         poi       date       day  predicted
   5710 Carter Lake 2025-07-01   Tuesday     111.60
   5710 Carter Lake 2025-07-02 Wednesday     118.80
   5710 Carter Lake 2025-07-03  Thursday     108.00
   5710 Carter Lake 2025-07-04    Friday     111.60
   5710 Carter Lake 2025-07-05  Saturday     128.34
   5710 Carter Lake 2025-07-06    Sunday     128.34
   5710 Carter Lake 2025-07-07    Monday     118.80


In [27]:
# Re-run with lighter model settings to finish quickly within the time limit

import os
import numpy as np
import pandas as pd
from datetime import timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

CSV_PATH = "C:/Users/israt/HD lab/My Project/Dataset/Airsage Dataset/NebraskaLakes_Cleaned_round.csv"

# ---------- Load ----------
df = pd.read_csv(CSV_PATH, low_memory=True)
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")
df = df.dropna(subset=["date"]).copy()
df["count"] = pd.to_numeric(df["count"], errors="coerce")
df = df.dropna(subset=["count"]).copy()

if "poi_id" not in df.columns:
    df["poi_id"] = pd.factorize(df["poi"].astype(str) if "poi" in df.columns else np.arange(len(df)))[0]
df["poi_id"] = pd.to_numeric(df["poi_id"], errors="coerce").fillna(-1).astype(int)

def add_calendar_feats(d):
    d = d.copy()
    dt = d["date"]
    d["month"] = dt.dt.month.astype(np.int16)
    d["dow"] = dt.dt.weekday.astype(np.int16)
    d["day"] = dt.dt.day.astype(np.int16)
    d["week"] = dt.dt.isocalendar().week.astype(np.int16)
    d["is_weekend"] = (d["dow"] >= 5).astype(np.int8)
    return d

def add_lags(d, lags=(7, 14)):
    d = d.sort_values("date").copy()
    for L in lags:
        d[f"lag{L}"] = d["count"].shift(L)
    return d

def make_numeric_feature_list(d, target="count"):
    num_cols = d.select_dtypes(include=[np.number]).columns.tolist()
    if target in num_cols:
        num_cols.remove(target)
    return num_cols

def fill_missing_features(X):
    med = np.nanmedian(X, axis=0)
    return np.where(np.isnan(X), med, X)

def train_poi_model(df_all, poi_id, target="count"):
    d = df_all[df_all["poi_id"] == poi_id].sort_values("date").copy()
    if d.empty:
        raise ValueError(f"No rows for poi_id={poi_id}.")
    d = add_calendar_feats(d)
    d = add_lags(d, lags=(7, 14))
    d_train = d.dropna(subset=["lag7", "lag14"]).copy()
    used_lags = True
    if d_train.empty:
        d_train = d.copy()
        used_lags = False

    feat_cols = make_numeric_feature_list(d_train, target=target)
    X = d_train[feat_cols].to_numpy(dtype=np.float32)
    y = d_train[target].to_numpy(dtype=np.float32)
    X = fill_missing_features(X)

    rf = RandomForestRegressor(
        n_estimators=120, max_depth=12, max_features="sqrt",
        random_state=42, n_jobs=-1
    ).fit(X, y)

    cut = max(1, int(0.8 * len(d_train)))
    r2 = mae = np.nan
    if len(d_train) - cut >= 5:
        X_te, y_te = X[cut:], y[cut:]
        y_hat = rf.predict(X_te)
        r2 = r2_score(y_te, y_hat)
        mae = mean_absolute_error(y_te, y_hat)

    meta = {
        "feat_cols": feat_cols,
        "used_lags": used_lags,
        "last_date": d["date"].max(),
        "poi_name": d["poi"].iloc[0] if "poi" in d.columns else str(poi_id),
        "r2_tail20": float(r2) if r2==r2 else None,
        "mae_tail20": float(mae) if mae==mae else None,
        "train_rows": int(len(d_train)),
    }
    return rf, meta, d

def forecast_future(df_all, poi_id, days=14, start_date=None, target="count"):
    model, meta, dfull = train_poi_model(df_all, poi_id, target=target)
    feat_cols = meta["feat_cols"]
    used_lags = meta["used_lags"]
    last_date = meta["last_date"]

    if start_date is None:
        start = last_date + timedelta(days=1)
    else:
        start = pd.to_datetime(str(start_date), format="%Y%m%d", errors="coerce")
        if pd.isna(start):
            raise ValueError("start_date must be YYYYMMDD or datetime-compatible")

    future_dates = pd.date_range(start, periods=days, freq="D")
    hist = dfull.sort_values("date")[["date", "count"]].copy()

    last_row = dfull.sort_values("date").iloc[-1:].copy()
    template_vals = last_row.select_dtypes(include=[np.number]).iloc[0].to_dict()

    out_rows = []
    for dt in future_dates:
        row = {"date": dt, "poi_id": int(poi_id)}
        row["month"] = int(dt.month)
        row["dow"] = int(dt.weekday())
        row["day"] = int(dt.day)
        row["week"] = int(dt.isocalendar().week)
        row["is_weekend"] = int(row["dow"] >= 5)
        for c, v in template_vals.items():
            if c not in ("count", "month", "dow", "day", "week", "is_weekend"):
                row[c] = float(v) if pd.notna(v) else np.nan
        out_rows.append(row)

    fut = pd.DataFrame(out_rows)

    # Recursive lag-aware prediction (fast; only 14 iterations by default)
    hist2 = hist.copy()
    preds = []
    for dt in future_dates:
        if used_lags:
            for L in (7, 14):
                lag_dt = dt - timedelta(days=L)
                m = hist2.loc[hist2["date"].eq(lag_dt), "count"]
                fut.loc[fut["date"].eq(dt), f"lag{L}"] = float(m.iloc[0]) if len(m) else np.nan
        xr = fut.loc[fut["date"].eq(dt), feat_cols].to_numpy(dtype=np.float32)
        xr = fill_missing_features(xr)
        yhat = float(model.predict(xr)[0])
        preds.append(yhat)
        hist2 = pd.concat([hist2, pd.DataFrame({"date":[dt], "count":[yhat]})], ignore_index=True)

    fut["predicted"] = preds
    fut = fut.sort_values("date").copy()
    fut["date_str"] = fut["date"].dt.strftime("%Y-%m-%d")
    fut["weekday"] = fut["date"].dt.day_name()

    cols_show = ["date_str", "weekday", "predicted"]
    for c in ["month", "dow", "is_weekend", "lag7", "lag14"]:
        if c in fut.columns and c not in cols_show:
            cols_show.append(c)

    poi_name = df[df["poi_id"] == poi_id]["poi"].iloc[0] if "poi" in df.columns and not df[df["poi_id"] == poi_id].empty else str(poi_id)
    safe_poi = "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in str(poi_name))
    out_path = f"/mnt/data/predictions_{safe_poi}_{future_dates[0].strftime('%Y%m%d')}_{days}d.csv"
    fut.to_csv(out_path, index=False)

    info = {
        "poi_id": poi_id,
        "poi_name": poi_name,
        "start_date": future_dates[0].strftime("%Y-%m-%d"),
        "days": days,
        "r2_tail20": meta["r2_tail20"],
        "mae_tail20": meta["mae_tail20"],
        "train_rows": meta["train_rows"],
        "used_lags": used_lags,
        "csv_path": out_path
    }
    return fut[cols_show], info

# Pick POI with most rows to demo
poicounts = df.groupby(["poi_id", "poi"] if "poi" in df.columns else ["poi_id"]).size().reset_index(name="n")
top_row = poicounts.sort_values("n", ascending=False).iloc[0]
demo_poi_id = int(top_row["poi_id"])
demo_name = str(top_row["poi"]) if "poi" in top_row else str(demo_poi_id)

demo_preds, demo_info = forecast_future(df, poi_id=demo_poi_id, days=14)

print("Demo POI:", demo_poi_id, "-", demo_name)
print("History rows:", demo_info["train_rows"], "| Tail-20% R²:", demo_info["r2_tail20"], "| MAE:", demo_info["mae_tail20"])
print("Forecast start:", demo_info["start_date"], "| Days:", demo_info["days"])
print("\nFirst few predictions:")
print(demo_preds.head(10).to_string(index=False))

print("\nSaved CSV:", demo_info["csv_path"])


C:\Users\israt\AppData\Local\Temp\ipykernel_13232\1230481685.py:13: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH, low_memory=True)


Demo POI: 5710 - Carter Lake
History rows: 11426 | Tail-20% R²: 0.9941983967210701 | MAE: 26.747222543799285
Forecast start: 2025-01-01 | Days: 14

First few predictions:
  date_str   weekday   predicted  month  dow  is_weekend        lag7  lag14
2025-01-01 Wednesday 1251.126786      1    2           0 1791.000000  948.0
2025-01-02  Thursday 1251.126786      1    3           0 1549.000000  948.0
2025-01-03    Friday 1251.479156      1    4           0  586.000000 1835.0
2025-01-04  Saturday 1247.333323      1    5           1  948.000000 1266.0
2025-01-05    Sunday 1241.456239      1    6           1  948.000000  948.0
2025-01-06    Monday 1247.466369      1    0           0  607.000000  948.0
2025-01-07   Tuesday 1245.716369      1    1           0  607.000000  994.0
2025-01-08 Wednesday 1244.083036      1    2           0 1251.126786 1791.0
2025-01-09  Thursday 1244.083036      1    3           0 1251.126786 1549.0
2025-01-10    Friday 1245.835406      1    4           0 1251.479156 

In [ ]:
# Re-run with lighter model settings to finish quickly within the time limit

import os
import numpy as np
import pandas as pd
from datetime import timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
CSV_PATH = "C:/Users/israt/HD lab/My Project/Dataset/Airsage Dataset/NebraskaLakes_Cleaned_round.csv"

# ---------- Load ----------
df = pd.read_csv(CSV_PATH, low_memory=True)
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")
df = df.dropna(subset=["date"]).copy()
df["count"] = pd.to_numeric(df["count"], errors="coerce")
df = df.dropna(subset=["count"]).copy()

if "poi_id" not in df.columns:
    df["poi_id"] = pd.factorize(df["poi"].astype(str) if "poi" in df.columns else np.arange(len(df)))[0]
df["poi_id"] = pd.to_numeric(df["poi_id"], errors="coerce").fillna(-1).astype(int)

def add_calendar_feats(d):
    d = d.copy()
    dt = d["date"]
    d["month"] = dt.dt.month.astype(np.int16)
    d["dow"] = dt.dt.weekday.astype(np.int16)
    d["day"] = dt.dt.day.astype(np.int16)
    d["week"] = dt.dt.isocalendar().week.astype(np.int16)
    d["is_weekend"] = (d["dow"] >= 5).astype(np.int8)
    return d

def add_lags(d, lags=(7, 14)):
    d = d.sort_values("date").copy()
    for L in lags:
        d[f"lag{L}"] = d["count"].shift(L)
    return d

def make_numeric_feature_list(d, target="count"):
    num_cols = d.select_dtypes(include=[np.number]).columns.tolist()
    if target in num_cols:
        num_cols.remove(target)
    return num_cols

def fill_missing_features(X):
    med = np.nanmedian(X, axis=0)
    return np.where(np.isnan(X), med, X)

def train_poi_model(df_all, poi_id, target="count"):
    d = df_all[df_all["poi_id"] == poi_id].sort_values("date").copy()
    if d.empty:
        raise ValueError(f"No rows for poi_id={poi_id}.")
    d = add_calendar_feats(d)
    d = add_lags(d, lags=(7, 14))
    d_train = d.dropna(subset=["lag7", "lag14"]).copy()
    used_lags = True
    if d_train.empty:
        d_train = d.copy()
        used_lags = False

    feat_cols = make_numeric_feature_list(d_train, target=target)
    X = d_train[feat_cols].to_numpy(dtype=np.float32)
    y = d_train[target].to_numpy(dtype=np.float32)
    X = fill_missing_features(X)

    rf = RandomForestRegressor(
        n_estimators=120, max_depth=12, max_features="sqrt",
        random_state=42, n_jobs=-1
    ).fit(X, y)

    cut = max(1, int(0.8 * len(d_train)))
    r2 = np.nan
    if len(d_train) - cut >= 5:
        X_te, y_te = X[cut:], y[cut:]
        y_hat = rf.predict(X_te)
        r2 = r2_score(y_te, y_hat)

    meta = {
        "feat_cols": feat_cols,
        "used_lags": used_lags,
        "last_date": d["date"].max(),
        "poi_name": d["poi"].iloc[0] if "poi" in d.columns else str(poi_id),
        "r2_tail20": float(r2) if r2==r2 else None,
        "train_rows": int(len(d_train)),
    }
    return rf, meta, d

def forecast_future(df_all, poi_id, days=14, start_date=None, target="count"):
    model, meta, dfull = train_poi_model(df_all, poi_id, target=target)
    feat_cols = meta["feat_cols"]
    used_lags = meta["used_lags"]
    last_date = meta["last_date"]

    if start_date is None:
        start = last_date + timedelta(days=1)
    else:
        start = pd.to_datetime(str(start_date), format="%Y%m%d", errors="coerce")
        if pd.isna(start):
            raise ValueError("start_date must be YYYYMMDD or datetime-compatible")

    future_dates = pd.date_range(start, periods=days, freq="D")
    hist = dfull.sort_values("date")[["date", "count"]].copy()

    last_row = dfull.sort_values("date").iloc[-1:].copy()
    template_vals = last_row.select_dtypes(include=[np.number]).iloc[0].to_dict()

    out_rows = []
    for dt in future_dates:
        row = {"date": dt, "poi_id": int(poi_id)}
        row["month"] = int(dt.month)
        row["dow"] = int(dt.weekday())
        row["day"] = int(dt.day)
        row["week"] = int(dt.isocalendar().week)
        row["is_weekend"] = int(row["dow"] >= 5)
        for c, v in template_vals.items():
            if c not in ("count", "month", "dow", "day", "week", "is_weekend"):
                row[c] = float(v) if pd.notna(v) else np.nan
        out_rows.append(row)

    fut = pd.DataFrame(out_rows)

    # Recursive lag-aware prediction
    hist2 = hist.copy()
    preds = []
    for dt in future_dates:
        if used_lags:
            for L in (7, 14):
                lag_dt = dt - timedelta(days=L)
                m = hist2.loc[hist2["date"].eq(lag_dt), "count"]
                fut.loc[fut["date"].eq(dt), f"lag{L}"] = float(m.iloc[0]) if len(m) else np.nan
        xr = fut.loc[fut["date"].eq(dt), feat_cols].to_numpy(dtype=np.float32)
        xr = fill_missing_features(xr)
        yhat = float(model.predict(xr)[0])
        preds.append(yhat)
        hist2 = pd.concat([hist2, pd.DataFrame({"date":[dt], "count":[yhat]})], ignore_index=True)

    fut["predicted"] = preds
    fut = fut.sort_values("date").copy()
    fut["date_str"] = fut["date"].dt.strftime("%Y-%m-%d")
    fut["weekday"] = fut["date"].dt.day_name()

    cols_show = ["date_str", "weekday", "predicted"]
    for c in ["month", "dow", "is_weekend", "lag7", "lag14"]:
        if c in fut.columns and c not in cols_show:
            cols_show.append(c)

    poi_name = df[df["poi_id"] == poi_id]["poi"].iloc[0] if "poi" in df.columns and not df[df["poi_id"] == poi_id].empty else str(poi_id)
    safe_poi = "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in str(poi_name))
    out_path = f"/mnt/data/predictions_{safe_poi}_{future_dates[0].strftime('%Y%m%d')}_{days}d.csv"
    fut.to_csv(out_path, index=False)

    info = {
        "poi_id": poi_id,
        "poi_name": poi_name,
        "start_date": future_dates[0].strftime("%Y-%m-%d"),
        "days": days,
        "r2_tail20": meta["r2_tail20"],   # <-- test accuracy (R²)
        "train_rows": meta["train_rows"],
        "used_lags": used_lags,
        "csv_path": out_path
    }
    return fut[cols_show], info

# Pick POI with most rows to demo
poicounts = df.groupby(["poi_id", "poi"] if "poi" in df.columns else ["poi_id"]).size().reset_index(name="n")
top_row = poicounts.sort_values("n", ascending=False).iloc[0]
demo_poi_id = int(top_row["poi_id"])
demo_name = str(top_row["poi"]) if "poi" in top_row else str(demo_poi_id)

demo_preds, demo_info = forecast_future(df, poi_id=demo_poi_id, days=14)

print("Demo POI:", demo_poi_id, "-", demo_name)
# Only print Test accuracy (R²)
print("Test accuracy (R²):", f"{demo_info['r2_tail20']:.4f}" if demo_info["r2_tail20"] is not None else "NA")
print("Forecast start:", demo_info["start_date"], "| Days:", demo_info["days"])
print("\nFirst few predictions:")
print(demo_preds.head(10).to_string(index=False))
print("\nSaved CSV:", demo_info["csv_path"])


C:\Users\israt\AppData\Local\Temp\ipykernel_13232\614333172.py:12: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH, low_memory=True)


Demo POI: 5710 - Carter Lake
Test accuracy (R²): 0.9942
Forecast start: 2025-01-01 | Days: 14

First few predictions:
  date_str   weekday   predicted  month  dow  is_weekend        lag7  lag14
2025-01-01 Wednesday 1251.126786      1    2           0 1791.000000  948.0
2025-01-02  Thursday 1251.126786      1    3           0 1549.000000  948.0
2025-01-03    Friday 1251.479156      1    4           0  586.000000 1835.0
2025-01-04  Saturday 1247.333323      1    5           1  948.000000 1266.0
2025-01-05    Sunday 1241.456239      1    6           1  948.000000  948.0
2025-01-06    Monday 1247.466369      1    0           0  607.000000  948.0
2025-01-07   Tuesday 1245.716369      1    1           0  607.000000  994.0
2025-01-08 Wednesday 1244.083036      1    2           0 1251.126786 1791.0
2025-01-09  Thursday 1244.083036      1    3           0 1251.126786 1549.0
2025-01-10    Friday 1245.835406      1    4           0 1251.479156  586.0

Saved CSV: /mnt/data/predictions_Carter_Lake_